# Faster-Whisper local ASR

In [14]:
import librosa
import numpy as np
import noisereduce as nr
import webrtcvad
import pyloudnorm as pyln
from scipy.signal import butter, lfilter
import collections
import numpy as np
import whisper
from faster_whisper import WhisperModel
import torch, torchaudio
import matplotlib.pyplot as plt
import time

## Full pieline

In [2]:
def load_audio(path, target_sr=16000):
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    audio = librosa.util.normalize(audio)
    return audio, target_sr

In [3]:
def reduce_noise(audio, sr):
    reduced = nr.reduce_noise(
        y=audio,
        sr=sr,
        stationary=False,
        prop_decrease=0.8
    )
    return reduced

In [4]:
def frame_generator(audio, frame_duration_ms, sr):
    frame_length = int(sr * frame_duration_ms / 1000)
    num_frames = len(audio) // frame_length
    for i in range(num_frames):
        start = i * frame_length
        end = start + frame_length
        yield audio[start:end]

def apply_vad(audio, sr, aggressiveness=2):
    vad = webrtcvad.Vad(aggressiveness)
    frame_duration = 30  # ms
    frame_length = int(sr * frame_duration / 1000)
    frames = list(frame_generator(audio, frame_duration, sr))
    voiced_audio = []

    for frame in frames:
        if len(frame) != frame_length:
            continue  # skip incomplete frames
        pcm_frame = (frame * 32768).astype(np.int16).tobytes()
        if vad.is_speech(pcm_frame, sr):
            voiced_audio.append(frame)

    if voiced_audio:
        return np.concatenate(voiced_audio)
    else:
        return np.array([], dtype=np.float32)    

In [5]:
def apply_agc(audio, target_rms=0.1, eps=1e-6):
    rms = np.sqrt(np.mean(audio**2))
    gain = target_rms / (rms + eps)
    audio = audio * gain
    return np.clip(audio, -1.0, 1.0)
# more advanced
def loudness_normalization(audio, sr, target_lufs=-23.0):
    meter = pyln.Meter(sr)
    loudness = meter.integrated_loudness(audio)
    normalized = pyln.normalize.loudness(audio, loudness, target_lufs)
    return normalized

In [6]:
def butter_bandpass(lowcut, highcut, sr, order=5):
    nyquist = 0.5 * sr
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

def apply_bandpass_filter(audio, sr, lowcut=100.0, highcut=6000.0):
    b, a = butter_bandpass(lowcut, highcut, sr)
    filtered = lfilter(b, a, audio)
    return filtered

In [7]:
def chunk_audio(audio, sr, chunk_size=30, overlap=2):
    chunk_len = chunk_size * sr
    overlap_len = overlap * sr
    chunks = []

    start = 0
    while start < len(audio):
        end = start + chunk_len
        chunks.append(audio[start:end])
        start += chunk_len - overlap_len

    return chunks

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# whisper
model = whisper.load_model("tiny")
def transcribe_chunk_w(chunk, lang="en", prompt=None):
    result = model.transcribe(
        chunk,
        language=lang,
        initial_prompt=prompt if prompt else ""
    )
    return result["text"]

# faster-whisper

model = WhisperModel("tiny", device=device)
def transcribe_chunk_fw(chunk, lang="en", prompt=None):
    segments, info = model.transcribe(
        chunk,
        beam_size=3,
        temperature=0.0,
        initial_prompt=prompt if prompt else "",
        condition_on_previous_text=False
    )
    return " ".join([seg.text for seg in segments])

In [ ]:
def asr_pipeline(audio_path, lang, prompt, faster=False):
    audio, sr = load_audio(audio_path)
    # 1. Bandpass filter
    audio = apply_bandpass_filter(audio, sr)
    # 2. Mild noise reduction
    audio = reduce_noise(audio, sr)
    # 3. Automatic gain control
    audio = apply_agc(audio)
    # 4. Voice Activity Detection
    audio = apply_vad(audio, sr)
    # 5. Chunking
    chunks = chunk_audio(audio, sr, chunk_size=20, overlap=1)

    full_text = ""
    for chunk in chunks:
        if faster:
            text = transcribe_chunk_fw(chunk, lang=lang, prompt=prompt)
        else:
            text = transcribe_chunk_w(chunk, lang=lang, prompt=prompt)
        full_text += text + " "

    return full_text.strip()


In [ ]:
# audio_file = "./test_files/eng.wav"
audio_file = "./test_files/rab-m3-n4.wav"
# audio_file = "./test_files/sor-m3-l4.wav"
# audio_file = "./test_files/pau-m3-d2.wav"

txt = asr_pipeline(audio_file, lang="it", prompt="Conversazione sul cibo e la cucina")
print(txt)

inselata pasta chutta, cosce da niente o limoncello


## Whisper / Faster-Whisper comparison

In [23]:
# audio_file = "./test_files/eng.wav"
audio_file = "./test_files/rab-m3-n4.wav"
audio = load_audio(audio_file)[0]

model_w = whisper.load_model("tiny")
model_fw = WhisperModel("tiny", device=device)

In [24]:
times_w, texts_w = [], []
times_fw, texts_fw = [], []

for _ in range(10):
    # whisper
    t0 = time.time()
    result = model_w.transcribe(
        audio,
        language="it",
    )
    times_w.append(time.time() - t0)
    texts_w.append(result["text"])

    # faster-whisper
    t1 = time.time()
    segments, info = model_fw.transcribe(
        audio,
        language="it",
        beam_size=3,
        temperature=0.0,
        condition_on_previous_text=False
    )
    times_fw.append(time.time() - t1)
    texts_fw.append(" ".join([seg.text for seg in segments]))

C:\Users\MATTIA\AppData\Roaming\Python\Python314\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [25]:
print(f"Whisper time mean: {np.mean(times_w):.2f} s - std: {np.std(times_w):.2f} s")
print(f"Faster-Whisper time mean: {np.mean(times_fw):.2f} s - std: {np.std(times_fw):.2f} s")

Whisper time mean: 0.58 s - std: 0.10 s
Faster-Whisper time mean: 0.01 s - std: 0.00 s


In [28]:
print(f"Whisper texts:")
print("".join(t+"\n" for t in texts_w))

print(f"Faster-Whisper texts:")
print("".join(t+"\n" for t in texts_fw))

Whisper texts:
 inselata pasta slytta, cosa da niente l'immuncello
 inselata pasta slytta, cosa da niente l'immuncello
 inselata pasta slytta, cosa da niente l'immuncello
 inselata pasta slytta, cosa da niente l'immuncello
 inselata pasta slytta, cosa da niente l'immuncello
 inselata pasta slytta, cosa da niente l'immuncello
 inselata pasta slytta, cosa da niente l'immuncello
 inselata pasta slytta, cosa da niente l'immuncello
 inselata pasta slytta, cosa da niente l'immuncello
 inselata pasta slytta, cosa da niente l'immuncello

Faster-Whisper texts:
 Inselato basta sulta, cosce da niente l'immuncello.
 Inselato basta sulta, cosce da niente l'immuncello.
 Inselato basta sulta, cosce da niente l'immuncello.
 Inselato basta sulta, cosce da niente l'immuncello.
 Inselato basta sulta, cosce da niente l'immuncello.
 Inselato basta sulta, cosce da niente l'immuncello.
 Inselato basta sulta, cosce da niente l'immuncello.
 Inselato basta sulta, cosce da niente l'immuncello.
 Inselato basta su